In [1]:
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
from sentence_transformers import SentenceTransformer
import chromadb
import os

project_root = Path.cwd().parent

# Load the private API key
load_dotenv(project_root / ".env")
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError("OPENAI_API_KEY was not found.")

openai_client = OpenAI(api_key=api_key)
del api_key

# Load the same embedding model
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

# Reconnect to the existing vector database
chroma_client = chromadb.PersistentClient(
    path=str(project_root / "data" / "chroma_db")
)

collection = chroma_client.get_collection(
    name="novamart_support"
)

print("Evaluation environment is ready.")
print(f"Knowledge-base chunks: {collection.count()}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Evaluation environment is ready.
Knowledge-base chunks: 27


In [3]:
def retrieve_chunks(question, number_of_results=3):
    question_embedding = embedding_model.encode(
        question,
        normalize_embeddings=True
    )

    results = collection.query(
        query_embeddings=[question_embedding.tolist()],
        n_results=number_of_results,
        include=["documents", "metadatas", "distances"]
    )

    retrieved_chunks = []

    for position in range(number_of_results):
        retrieved_chunks.append({
            "chunk_id": results["ids"][0][position],
            "content": results["documents"][0][position],
            "source": results["metadatas"][0][position]["source"],
            "similarity": 1 - results["distances"][0][position]
        })

    return retrieved_chunks

print("Retrieval function updated with chunk IDs.")

Retrieval function updated with chunk IDs.


In [4]:
retrieval_test_cases = [
    {
        "question": "Can I return an unused item after 30 days?",
        "expected_chunk_id": "returns_refunds.md_1"
    },
    {
        "question": "My product arrived broken. What should I provide?",
        "expected_chunk_id": "returns_refunds.md_2"
    },
    {
        "question": "How long does an approved refund take?",
        "expected_chunk_id": "returns_refunds.md_3"
    },
    {
        "question": "Can I return a personalised product?",
        "expected_chunk_id": "returns_refunds.md_4"
    },
    {
        "question": "How long does standard shipping take to a remote area?",
        "expected_chunk_id": "shipping_delivery.md_2"
    },
    {
        "question": "How quickly will express delivery reach a regional area?",
        "expected_chunk_id": "shipping_delivery.md_3"
    },
    {
        "question": "When will my tracking link start working?",
        "expected_chunk_id": "shipping_delivery.md_5"
    },
    {
        "question": "Tracking says delivered, but I cannot find my parcel.",
        "expected_chunk_id": "shipping_delivery.md_6"
    },
    {
        "question": "How long is the standard product warranty?",
        "expected_chunk_id": "product_warranty.md_1"
    },
    {
        "question": "Does the warranty cover accidental water damage?",
        "expected_chunk_id": "product_warranty.md_3"
    },
    {
        "question": "What evidence do I need for a warranty claim?",
        "expected_chunk_id": "product_warranty.md_4"
    },
    {
        "question": "Can I cancel an order that has already shipped?",
        "expected_chunk_id": "orders_payments.md_3"
    },
    {
        "question": "What should I do if my card payment is declined?",
        "expected_chunk_id": "orders_payments.md_5"
    },
    {
        "question": "Why do I see a pending charge on my card?",
        "expected_chunk_id": "orders_payments.md_6"
    },
    {
        "question": "Can I place an order without creating an account?",
        "expected_chunk_id": "account_privacy.md_1"
    },
    {
        "question": "How long is a password reset link valid?",
        "expected_chunk_id": "account_privacy.md_2"
    },
    {
        "question": "My account is locked after several login attempts.",
        "expected_chunk_id": "account_privacy.md_4"
    },
    {
        "question": "How long does account deletion take?",
        "expected_chunk_id": "account_privacy.md_6"
    }
]

print(f"Retrieval test cases created: {len(retrieval_test_cases)}")

Retrieval test cases created: 18


In [5]:
evaluation_results = []

for test_case in retrieval_test_cases:
    retrieved = retrieve_chunks(
        test_case["question"],
        number_of_results=3
    )

    retrieved_ids = [
        chunk["chunk_id"]
        for chunk in retrieved
    ]

    expected_id = test_case["expected_chunk_id"]

    top_1_correct = retrieved_ids[0] == expected_id
    top_3_correct = expected_id in retrieved_ids

    if top_3_correct:
        correct_rank = retrieved_ids.index(expected_id) + 1
        reciprocal_rank = 1 / correct_rank
    else:
        correct_rank = None
        reciprocal_rank = 0

    evaluation_results.append({
        "question": test_case["question"],
        "expected_chunk_id": expected_id,
        "retrieved_chunk_ids": retrieved_ids,
        "top_1_correct": top_1_correct,
        "top_3_correct": top_3_correct,
        "correct_rank": correct_rank,
        "reciprocal_rank": reciprocal_rank
    })

top_1_accuracy = sum(
    result["top_1_correct"]
    for result in evaluation_results
) / len(evaluation_results)

top_3_accuracy = sum(
    result["top_3_correct"]
    for result in evaluation_results
) / len(evaluation_results)

mean_reciprocal_rank = sum(
    result["reciprocal_rank"]
    for result in evaluation_results
) / len(evaluation_results)

print(f"Top-1 accuracy: {top_1_accuracy:.2%}")
print(f"Top-3 accuracy: {top_3_accuracy:.2%}")
print(f"Mean Reciprocal Rank: {mean_reciprocal_rank:.3f}")

Top-1 accuracy: 94.44%
Top-3 accuracy: 100.00%
Mean Reciprocal Rank: 0.972


In [6]:
failed_top_1_cases = [
    result
    for result in evaluation_results
    if not result["top_1_correct"]
]

print(f"Top-1 failures: {len(failed_top_1_cases)}")

for failure in failed_top_1_cases:
    print("\nQuestion:")
    print(failure["question"])

    print("\nExpected chunk:")
    print(failure["expected_chunk_id"])

    print("\nRetrieved ranking:")
    for rank, chunk_id in enumerate(
        failure["retrieved_chunk_ids"],
        start=1
    ):
        print(f"{rank}. {chunk_id}")

Top-1 failures: 1

Question:
Tracking says delivered, but I cannot find my parcel.

Expected chunk:
shipping_delivery.md_6

Retrieved ranking:
1. shipping_delivery.md_5
2. shipping_delivery.md_6
3. shipping_delivery.md_1


In [7]:
import json

evaluation_path = project_root / "data" / "evaluation"
evaluation_path.mkdir(parents=True, exist_ok=True)

retrieval_evaluation_output = {
    "number_of_test_cases": len(evaluation_results),
    "top_1_accuracy": top_1_accuracy,
    "top_3_accuracy": top_3_accuracy,
    "mean_reciprocal_rank": mean_reciprocal_rank,
    "results": evaluation_results
}

output_file = evaluation_path / "retrieval_evaluation_results.json"

with output_file.open("w", encoding="utf-8") as file:
    json.dump(
        retrieval_evaluation_output,
        file,
        ensure_ascii=False,
        indent=2
    )

print(f"Evaluation saved to: {output_file}")

Evaluation saved to: C:\Users\arunp\Desktop\rag-assistant\data\evaluation\retrieval_evaluation_results.json


In [8]:
def answer_question(question, number_of_results=3):
    retrieved_chunks = retrieve_chunks(
        question,
        number_of_results=number_of_results
    )

    context_sections = []

    for chunk in retrieved_chunks:
        context_sections.append(
            f"Source: {chunk['source']}\n"
            f"Content:\n{chunk['content']}"
        )

    context = "\n\n---\n\n".join(context_sections)

    response = openai_client.responses.create(
        model="gpt-6-luna",
        instructions=(
            "You are NovaMart Australia's customer support assistant. "
            "Answer using only the supplied policy context. "
            "Do not use outside knowledge or invent company policies. "
            "Cite only a source that directly supports your answer. "
            "If the context does not directly answer the customer's question, "
            "respond exactly with: "
            "'I cannot answer this from the available NovaMart policies.' "
            "Do not include a citation when giving this refusal. "
            "Otherwise, give a concise and helpful answer and cite the "
            "supporting source filename in square brackets."
        ),
        input=(
            f"Customer question:\n{question}\n\n"
            f"NovaMart policy context:\n{context}"
        ),
        max_output_tokens=300
    )

    return {
        "question": question,
        "answer": response.output_text,
        "retrieved_chunks": retrieved_chunks
    }

print("RAG function is ready for answer evaluation.")

RAG function is ready for answer evaluation.


In [9]:
answer_test_cases = [
    {
        "question": "My order arrived damaged. What should I do?",
        "answerable": True,
        "expected_source": "returns_refunds.md",
        "required_keywords": [
            "7 calendar days",
            "order number",
            "photos"
        ]
    },
    {
        "question": "How long is the password reset link valid?",
        "answerable": True,
        "expected_source": "account_privacy.md",
        "required_keywords": [
            "30 minutes"
        ]
    },
    {
        "question": "Does the warranty cover accidental damage?",
        "answerable": True,
        "expected_source": "product_warranty.md",
        "required_keywords": [
            "accidental damage",
            "not"
        ]
    },
    {
        "question": "Can I cancel an order after it has shipped?",
        "answerable": True,
        "expected_source": "orders_payments.md",
        "required_keywords": [
            "cannot be cancelled"
        ]
    },
    {
        "question": "How long does account deletion take?",
        "answerable": True,
        "expected_source": "account_privacy.md",
        "required_keywords": [
            "30 calendar days"
        ]
    },
    {
        "question": "Why is there a pending charge on my card?",
        "answerable": True,
        "expected_source": "orders_payments.md",
        "required_keywords": [
            "temporary authorisation",
            "3–5 business days"
        ]
    },
    {
        "question": "Does NovaMart match competitors' prices?",
        "answerable": False,
        "expected_source": None,
        "required_keywords": []
    },
    {
        "question": "Does NovaMart offer a student discount?",
        "answerable": False,
        "expected_source": None,
        "required_keywords": []
    },
    {
        "question": "What are NovaMart's phone support hours?",
        "answerable": False,
        "expected_source": None,
        "required_keywords": []
    }
]

print(f"Answer test cases created: {len(answer_test_cases)}")

Answer test cases created: 9


In [10]:
answer_evaluation_results = []

refusal_text = (
    "I cannot answer this from the available NovaMart policies."
)

for test_number, test_case in enumerate(answer_test_cases, start=1):
    print(
        f"Evaluating question {test_number} "
        f"of {len(answer_test_cases)}..."
    )

    result = answer_question(test_case["question"])
    answer = result["answer"].strip()

    normalized_answer = (
        answer.lower()
        .replace("–", "-")
        .replace("—", "-")
    )

    normalized_keywords = [
        keyword.lower()
        .replace("–", "-")
        .replace("—", "-")
        for keyword in test_case["required_keywords"]
    ]

    if test_case["answerable"]:
        facts_correct = all(
            keyword in normalized_answer
            for keyword in normalized_keywords
        )

        expected_citation = (
            f"[{test_case['expected_source']}]"
        )

        citation_correct = (
            expected_citation in answer
        )

        refusal_correct = None
        overall_correct = facts_correct and citation_correct

    else:
        facts_correct = None
        citation_correct = "[" not in answer and "]" not in answer
        refusal_correct = answer == refusal_text
        overall_correct = refusal_correct and citation_correct

    answer_evaluation_results.append({
        "question": test_case["question"],
        "answerable": test_case["answerable"],
        "generated_answer": answer,
        "facts_correct": facts_correct,
        "citation_correct": citation_correct,
        "refusal_correct": refusal_correct,
        "overall_correct": overall_correct
    })

print("\nAnswer evaluation complete.")

Evaluating question 1 of 9...
Evaluating question 2 of 9...
Evaluating question 3 of 9...
Evaluating question 4 of 9...
Evaluating question 5 of 9...
Evaluating question 6 of 9...
Evaluating question 7 of 9...
Evaluating question 8 of 9...
Evaluating question 9 of 9...

Answer evaluation complete.


In [11]:
supported_results = [
    result
    for result in answer_evaluation_results
    if result["answerable"]
]

unsupported_results = [
    result
    for result in answer_evaluation_results
    if not result["answerable"]
]

fact_accuracy = sum(
    result["facts_correct"]
    for result in supported_results
) / len(supported_results)

citation_accuracy = sum(
    result["citation_correct"]
    for result in supported_results
) / len(supported_results)

refusal_accuracy = sum(
    result["refusal_correct"]
    for result in unsupported_results
) / len(unsupported_results)

overall_accuracy = sum(
    result["overall_correct"]
    for result in answer_evaluation_results
) / len(answer_evaluation_results)

print(f"Supported-answer fact accuracy: {fact_accuracy:.2%}")
print(f"Supported-answer citation accuracy: {citation_accuracy:.2%}")
print(f"Unsupported-question refusal accuracy: {refusal_accuracy:.2%}")
print(f"Overall answer accuracy: {overall_accuracy:.2%}")

failed_answer_cases = [
    result
    for result in answer_evaluation_results
    if not result["overall_correct"]
]

print(f"\nFailed answer cases: {len(failed_answer_cases)}")

for failure in failed_answer_cases:
    print("\nQuestion:")
    print(failure["question"])

    print("\nGenerated answer:")
    print(failure["generated_answer"])

    print("\nChecks:")
    print("Facts correct:", failure["facts_correct"])
    print("Citation correct:", failure["citation_correct"])
    print("Refusal correct:", failure["refusal_correct"])

Supported-answer fact accuracy: 100.00%
Supported-answer citation accuracy: 100.00%
Unsupported-question refusal accuracy: 100.00%
Overall answer accuracy: 100.00%

Failed answer cases: 0


In [12]:
answer_evaluation_output = {
    "number_of_test_cases": len(answer_evaluation_results),
    "supported_test_cases": len(supported_results),
    "unsupported_test_cases": len(unsupported_results),
    "fact_accuracy": fact_accuracy,
    "citation_accuracy": citation_accuracy,
    "refusal_accuracy": refusal_accuracy,
    "overall_accuracy": overall_accuracy,
    "results": answer_evaluation_results
}

answer_output_file = (
    evaluation_path / "answer_evaluation_results.json"
)

with answer_output_file.open("w", encoding="utf-8") as file:
    json.dump(
        answer_evaluation_output,
        file,
        ensure_ascii=False,
        indent=2
    )

print(f"Answer evaluation saved to: {answer_output_file}")

Answer evaluation saved to: C:\Users\arunp\Desktop\rag-assistant\data\evaluation\answer_evaluation_results.json


In [13]:
src_init_file = project_root / "src" / "__init__.py"
src_init_file.touch(exist_ok=True)

print(f"Created: {src_init_file}")

Created: C:\Users\arunp\Desktop\rag-assistant\src\__init__.py


In [14]:
%%writefile ../src/rag_pipeline.py

from functools import lru_cache
from pathlib import Path
import os

import chromadb
from dotenv import load_dotenv
from openai import OpenAI
from sentence_transformers import SentenceTransformer


PROJECT_ROOT = Path(__file__).resolve().parent.parent
CHROMA_PATH = PROJECT_ROOT / "data" / "chroma_db"
COLLECTION_NAME = "novamart_support"
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
LLM_MODEL_NAME = "gpt-6-luna"

load_dotenv(PROJECT_ROOT / ".env")


@lru_cache(maxsize=1)
def get_embedding_model():
    return SentenceTransformer(EMBEDDING_MODEL_NAME)


@lru_cache(maxsize=1)
def get_collection():
    chroma_client = chromadb.PersistentClient(
        path=str(CHROMA_PATH)
    )

    return chroma_client.get_collection(
        name=COLLECTION_NAME
    )


@lru_cache(maxsize=1)
def get_openai_client():
    api_key = os.getenv("OPENAI_API_KEY")

    if not api_key:
        raise ValueError(
            "OPENAI_API_KEY was not found. "
            "Add it to the project's .env file."
        )

    return OpenAI(api_key=api_key)


def retrieve_chunks(question, number_of_results=3):
    embedding_model = get_embedding_model()
    collection = get_collection()

    question_embedding = embedding_model.encode(
        question,
        normalize_embeddings=True
    )

    results = collection.query(
        query_embeddings=[question_embedding.tolist()],
        n_results=number_of_results,
        include=["documents", "metadatas", "distances"]
    )

    retrieved_chunks = []

    for position in range(number_of_results):
        retrieved_chunks.append({
            "chunk_id": results["ids"][0][position],
            "content": results["documents"][0][position],
            "source": results["metadatas"][0][position]["source"],
            "similarity": 1 - results["distances"][0][position]
        })

    return retrieved_chunks


def answer_question(question, number_of_results=3):
    retrieved_chunks = retrieve_chunks(
        question,
        number_of_results=number_of_results
    )

    context_sections = []

    for chunk in retrieved_chunks:
        context_sections.append(
            f"Source: {chunk['source']}\n"
            f"Content:\n{chunk['content']}"
        )

    context = "\n\n---\n\n".join(context_sections)

    openai_client = get_openai_client()

    response = openai_client.responses.create(
        model=LLM_MODEL_NAME,
        instructions=(
            "You are NovaMart Australia's customer support assistant. "
            "Answer using only the supplied policy context. "
            "Do not use outside knowledge or invent company policies. "
            "Cite only a source that directly supports your answer. "
            "If the context does not directly answer the customer's "
            "question, respond exactly with: "
            "'I cannot answer this from the available NovaMart policies.' "
            "Do not include a citation when giving this refusal. "
            "Otherwise, give a concise and helpful answer and cite the "
            "supporting source filename in square brackets."
        ),
        input=(
            f"Customer question:\n{question}\n\n"
            f"NovaMart policy context:\n{context}"
        ),
        max_output_tokens=300
    )

    return {
        "question": question,
        "answer": response.output_text,
        "retrieved_chunks": retrieved_chunks
    }

Writing ../src/rag_pipeline.py


In [15]:
import sys

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.rag_pipeline import retrieve_chunks as module_retrieve

module_results = module_retrieve(
    "How long is the password reset link valid?"
)

print("Module imported successfully.")
print("Top retrieved chunk:", module_results[0]["chunk_id"])
print("Source:", module_results[0]["source"])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Module imported successfully.
Top retrieved chunk: account_privacy.md_2
Source: account_privacy.md


In [16]:
%pip install -U streamlit

   ---------------------------------------- 0.0/10.1 MB ? eta -:--:--
   --------------------- ------------------ 5.5/10.1 MB 33.6 MB/s eta 0:00:01
   ---------------------------------------- 10.1/10.1 MB 30.0 MB/s  0:00:00
   ---------------------------------------- 0.0/797.2 kB ? eta -:--:--
   ---------------------------------------- 797.2/797.2 kB 33.4 MB/s  0:00:00
   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   ---------------------------------------  9.7/9.9 MB 60.1 MB/s eta 0:00:01
   ---------------------------------------- 9.9/9.9 MB 40.8 MB/s  0:00:00
   ---------------------------------------- 0.0/7.2 MB ? eta -:--:--
   ---------------------------------------- 7.2/7.2 MB 37.1 MB/s  0:00:00
   ---------------------------------------- 0.0/27.9 MB ? eta -:--:--
   ------------------- -------------------- 13.6/27.9 MB 71.5 MB/s eta 0:00:01
   ---------------------------------------  27.3/27.9 MB 69.2 MB/s eta 0:00:01
   ---------------------------------

In [17]:
%%writefile ../app.py

import streamlit as st

from src.rag_pipeline import answer_question


st.set_page_config(
    page_title="NovaMart Support Assistant",
    page_icon="🛍️",
    layout="centered"
)

st.title("NovaMart Support Assistant")
st.caption(
    "A retrieval-augmented customer support assistant that answers "
    "questions using NovaMart's policy documents."
)

with st.sidebar:
    st.header("About this project")

    st.markdown(
        """
        This assistant uses:

        - Local sentence-transformer embeddings
        - ChromaDB semantic retrieval
        - OpenAI answer generation
        - Source attribution
        - Grounded refusal for unsupported questions
        """
    )

    st.header("Example questions")

    st.markdown(
        """
        - Can I return an unused item after 30 days?
        - My order arrived damaged. What should I do?
        - How long is the product warranty?
        - Can I cancel an order after it ships?
        - How long is a password reset link valid?
        """
    )

with st.form("customer_question_form"):
    question = st.text_area(
        "Ask a customer-support question",
        placeholder=(
            "Example: My order arrived damaged. What should I do?"
        ),
        height=120
    )

    submitted = st.form_submit_button(
        "Ask NovaMart",
        use_container_width=True
    )

if submitted:
    if not question.strip():
        st.warning("Please enter a question.")

    else:
        with st.spinner("Searching NovaMart policies..."):
            try:
                result = answer_question(question.strip())

            except Exception as error:
                st.error(
                    "The assistant could not generate an answer. "
                    "Please check the application setup and try again."
                )

                with st.expander("Technical details"):
                    st.code(str(error))

            else:
                st.subheader("Answer")
                st.write(result["answer"])

                st.subheader("Retrieved evidence")

                st.caption(
                    "These are the policy sections retrieved before "
                    "the answer was generated."
                )

                for position, chunk in enumerate(
                    result["retrieved_chunks"],
                    start=1
                ):
                    label = (
                        f"{position}. {chunk['source']} "
                        f"— similarity {chunk['similarity']:.3f}"
                    )

                    with st.expander(label):
                        st.markdown(chunk["content"])

Writing ../app.py


In [2]:
from pathlib import Path

project_root = Path.cwd().parent

print(project_root)

C:\Users\arunp\Desktop\rag-assistant


In [3]:
app_file = project_root / "app.py"
app_code = app_file.read_text(encoding="utf-8")

old_code = '''                st.subheader("Retrieved evidence")

                st.caption(
                    "These are the policy sections retrieved before "
                    "the answer was generated."
                )

                for position, chunk in enumerate(
                    result["retrieved_chunks"],
                    start=1
                ):
                    label = (
                        f"{position}. {chunk['source']} "
                        f"— similarity {chunk['similarity']:.3f}"
                    )

                    with st.expander(label):
                        st.markdown(chunk["content"])
'''

new_code = '''                refusal_text = (
                    "I cannot answer this from the available "
                    "NovaMart policies."
                )

                if result["answer"].strip() == refusal_text:
                    st.info(
                        "No directly supported NovaMart policy was found."
                    )

                else:
                    st.subheader("Retrieved evidence")

                    st.caption(
                        "These are the policy sections retrieved before "
                        "the answer was generated."
                    )

                    for position, chunk in enumerate(
                        result["retrieved_chunks"],
                        start=1
                    ):
                        label = (
                            f"{position}. {chunk['source']} "
                            f"— similarity {chunk['similarity']:.3f}"
                        )

                        with st.expander(label):
                            st.markdown(chunk["content"])
'''

if old_code not in app_code:
    raise ValueError("The expected app code was not found.")

updated_app_code = app_code.replace(old_code, new_code)
app_file.write_text(updated_app_code, encoding="utf-8")

print("Streamlit app updated.")

Streamlit app updated.


In [4]:
from importlib.metadata import version

packages = [
    "streamlit",
    "openai",
    "python-dotenv",
    "sentence-transformers",
    "chromadb"
]

for package in packages:
    print(f"{package}=={version(package)}")

streamlit==1.64.0
openai==3.22.1
python-dotenv==1.2.3
sentence-transformers==6.1.0
chromadb==1.5.9


In [5]:
requirements = """streamlit==1.64.0
openai==3.22.1
python-dotenv==1.2.3
sentence-transformers==6.1.0
chromadb==1.5.9
"""

requirements_file = project_root / "requirements.txt"
requirements_file.write_text(requirements, encoding="utf-8")

print(f"Created: {requirements_file}")

Created: C:\Users\arunp\Desktop\rag-assistant\requirements.txt


In [6]:
gitignore_content = """.env
__pycache__/
*.py[cod]
.ipynb_checkpoints/
.DS_Store
"""

env_example_content = """OPENAI_API_KEY=your_openai_api_key_here
"""

(project_root / ".gitignore").write_text(
    gitignore_content,
    encoding="utf-8"
)

(project_root / ".env.example").write_text(
    env_example_content,
    encoding="utf-8"
)

print("Updated .gitignore")
print("Created .env.example")

Updated .gitignore
Created .env.example


In [7]:
%%writefile ../README.md

# NovaMart AI Customer Support Assistant

An end-to-end retrieval-augmented generation (RAG) application that answers customer-support questions using a fictional Australian e-commerce knowledge base.

The system retrieves relevant policy sections from ChromaDB and provides them to an OpenAI language model, which generates a grounded answer with source attribution.

## Features

- Customer-support knowledge base covering returns, shipping, warranties, orders, payments, accounts and privacy
- Local embeddings using `sentence-transformers/all-MiniLM-L6-v2`
- Persistent vector storage using ChromaDB
- Semantic retrieval of relevant policy sections
- Grounded answer generation using the OpenAI Responses API
- Source citations in generated answers
- Refusal behaviour for unsupported questions
- Retrieval and answer-quality evaluation
- Interactive Streamlit interface

## Architecture

```text
Policy documents
      ↓
Document loading and section-based chunking
      ↓
Sentence Transformer embeddings
      ↓
ChromaDB vector database
      ↓
Customer question → semantic retrieval
      ↓
Top policy sections + customer question
      ↓
OpenAI language model
      ↓
Grounded answer with source citation

Writing ../README.md
